<a href="https://colab.research.google.com/github/ife0042/odoo_import_pipeline/blob/main/Odoo_import_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !ls /content/drive/MyDrive

In [44]:
INVENTORY_GOOGLE_SHEET = "Inventory Store Tracker"
RAW_INVENTORY_TAB_NAME = "Inventory Listing"
PRODUCT_IMAGES_DIR_PATH = ""
ENV = "test"

In [83]:
# Constants
SALES_TAX_EXTERNAL_ID = None # 'account.1_l10n_ng_vat_out_7_5'
PURCHASE_TAX_EXTERNAL_ID = None # 'account.1_l10n_ng_vat_in_7_5'
# Define the Google Drive folder path for product images
IMAGE_FOLDER_PATH = '/content/drive/MyDrive/product_images/'

In [82]:
# Import Libraries
from google.colab import auth
import pandas as pd
from google.auth import default
import gspread
import os

pd.set_option('display.width', 1000)

In [57]:
# Authentication and initialization of Sheet reader
auth.authenticate_user()
# Get credentials from google.auth.default()
creds, _ = default()
# Initialize the gspread client with the obtained credentials
gc = gspread.authorize(creds)

# Read Raw Inventory WorkSheet
Read the `raw_inventory_worksheet` into a pandas DataFrame.

In [64]:
# Open the Google Sheet and get the worksheet
inventory_google_sheet_obj = gc.open(INVENTORY_GOOGLE_SHEET)
raw_inventory_worksheet = inventory_google_sheet_obj.worksheet(RAW_INVENTORY_TAB_NAME)

# Get all records from the worksheet
raw_inventory_data = raw_inventory_worksheet.get_all_records()

# Create a pandas DataFrame
raw_inventory_df = pd.DataFrame(raw_inventory_data)

# Display the first 5 rows of the DataFrame
print("First 5 rows of the inventory DataFrame:")
print(raw_inventory_df.head())

First 5 rows of the inventory DataFrame:
    Name       Category Original Price Sales Price    Variant Quantity Cost Price Total Cost Price Sold (Total Amount) Sold (Total Quantity)
0  AA001  Women / Heels          12000       12000   Black-39        1       7500             7500               12000                     1
1                                                    Black-40        1       7500             7500                                          
2                                                    Black-41        1       7500             7500                                          
3                                                   Purple-37        1       7500             7500                                          
4                                                   Purple-39        2       7500            15000                                          


## Rename fields: "Name", "Category"

In [89]:
raw_inventory_df.rename(columns={'Name': 'Product Name'}, inplace=True)
raw_inventory_df.rename(columns={'Category': 'Website Product Category'}, inplace=True)
raw_inventory_df.rename(columns={'Quantity': 'Variant Quantity on Hand'}, inplace=True)

# Display the first 5 rows of the DataFrame
print("First 5 rows of the inventory DataFrame:")
print(raw_inventory_df.head())

First 5 rows of the inventory DataFrame:
  Product Name Website Product Category  Original Price  Sales Price    Variant Variant Quantity on Hand Cost Price Total Cost Price Sold (Total Amount) Sold (Total Quantity) Product Category  Is Published  Purchase  Sales  Track Inventory    Invoicing Policy     Tracking      Product Type             External ID  Compare to Price Sales Taxes / External ID Purchase Taxes / External ID Variant Colour Variant Size Variant Quantity on Hand
0        AA001                    Heels           12000        12000   Black-39                        1       7500             7500               12000                     1            Goods          True      True   True             True  Ordered Quantities  No Tracking  Storable Product  __imported__AA001_test               NaN                      None                         None          Black           39                        1
1        AA001                    Heels           12000        12000   Black-

## Forward fill fields

In [70]:
raw_inventory_df['Product Name'] = raw_inventory_df['Product Name'].replace('', method='ffill')
raw_inventory_df['Website Product Category'] = raw_inventory_df['Website Product Category'].replace('', method='ffill')
raw_inventory_df['Original Price'] = raw_inventory_df['Original Price'].replace('', method='ffill')
raw_inventory_df['Sales Price'] = raw_inventory_df['Sales Price'].replace('', method='ffill')

# Display the first 5 rows of the DataFrame
print("First 5 rows of the inventory DataFrame:")
print(raw_inventory_df.head())

First 5 rows of the inventory DataFrame:
  Product Name Website Product Category Original Price Sales Price    Variant Quantity Cost Price Total Cost Price Sold (Total Amount) Sold (Total Quantity)
0        AA001            Women / Heels          12000       12000   Black-39        1       7500             7500               12000                     1
1        AA001            Women / Heels          12000       12000   Black-40        1       7500             7500                                          
2        AA001            Women / Heels          12000       12000   Black-41        1       7500             7500                                          
3        AA001            Women / Heels          12000       12000  Purple-37        1       7500             7500                                          
4        AA001            Women / Heels          12000       12000  Purple-39        2       7500            15000                                          


/tmp/ipython-input-2244426526.py:1: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  raw_inventory_df['Product Name'] = raw_inventory_df['Product Name'].replace('', method='ffill')
/tmp/ipython-input-2244426526.py:2: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  raw_inventory_df['Website Product Category'] = raw_inventory_df['Website Product Category'].replace('', method='ffill')
/tmp/ipython-input-2244426526.py:3: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  raw_inventory_df['Original Price'] = raw_inventory_df['Original Price'].replace('', method='ffill')
/tmp/ipython-input-2244426526.py:4: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  raw_inventory_df['Sales Price'] = raw_inventory_df['Sales Price'].replace('', method='ffill')


## Initial Column Mapping and Static Assignments

### Subtask:
Map existing columns and set static values for new columns required for the product data structure.


In [71]:
raw_inventory_df['Product Category'] = 'Goods'
raw_inventory_df['Is Published'] = True
raw_inventory_df['Purchase'] = True
raw_inventory_df['Sales'] = True
raw_inventory_df['Track Inventory'] = True
raw_inventory_df['Invoicing Policy'] = 'Ordered Quantities'
raw_inventory_df['Tracking'] = 'No Tracking'
raw_inventory_df['Product Type'] = 'Storable Product'

# print("First 5 rows of the inventory DataFrame after adding new columns:")
# print(raw_inventory_df.head())

## Generate Product External ID

### Subtask:
Create the 'External ID' column based on 'Product Name' and an environment variable (e.g., 'test' or 'prod').


In [76]:
raw_inventory_df['External ID'] = '__imported__' + raw_inventory_df['Product Name'].astype(str) + ('' if ENV=='prod' else '_' + ENV)

# Display the first 5 rows of the DataFrame
print("First 5 rows of the inventory DataFrame after filling 'Name' and regenerating 'External ID':")
print(raw_inventory_df.head())

First 5 rows of the inventory DataFrame after filling 'Name' and regenerating 'External ID':
  Product Name Website Product Category Original Price Sales Price    Variant Quantity Cost Price Total Cost Price Sold (Total Amount) Sold (Total Quantity) Product Category  Is Published  Purchase  Sales  Track Inventory    Invoicing Policy     Tracking      Product Type             External ID
0        AA001            Women / Heels          12000       12000   Black-39        1       7500             7500               12000                     1            Goods          True      True   True             True  Ordered Quantities  No Tracking  Storable Product  __imported__AA001_test
1        AA001            Women / Heels          12000       12000   Black-40        1       7500             7500                                                      Goods          True      True   True             True  Ordered Quantities  No Tracking  Storable Product  __imported__AA001_test
2        AA001  

## Process Website Product Category

### Subtask:
Transform the 'Category' column to 'Website Product Category' by splitting values by slash and selecting the last part.


In [77]:
raw_inventory_df['Website Product Category'] = raw_inventory_df['Website Product Category'].replace('', np.nan)
raw_inventory_df['Website Product Category'] = raw_inventory_df['Website Product Category'].apply(lambda x: x.split('/')[-1].strip())

print("First 5 rows of the inventory DataFrame with 'Website Product Category':")
print(raw_inventory_df.head())

First 5 rows of the inventory DataFrame with 'Website Product Category':
  Product Name Website Product Category Original Price Sales Price    Variant Quantity Cost Price Total Cost Price Sold (Total Amount) Sold (Total Quantity) Product Category  Is Published  Purchase  Sales  Track Inventory    Invoicing Policy     Tracking      Product Type             External ID
0        AA001                    Heels          12000       12000   Black-39        1       7500             7500               12000                     1            Goods          True      True   True             True  Ordered Quantities  No Tracking  Storable Product  __imported__AA001_test
1        AA001                    Heels          12000       12000   Black-40        1       7500             7500                                                      Goods          True      True   True             True  Ordered Quantities  No Tracking  Storable Product  __imported__AA001_test
2        AA001                    He

## Calculate Compare to Price

### Subtask:
Implement logic to assign a value to 'Compare to Price' if 'Original Price' is higher than 'Sales Price'.


**Reasoning**:
To implement the logic for 'Compare to Price', I need to first ensure 'Original Price' and 'Sales Price' are numeric, then initialize 'Compare to Price' as empty, and finally apply conditional logic to populate 'Compare to Price' where 'Original Price' is greater than 'Sales Price', displaying the first 5 rows to confirm.



In [78]:
raw_inventory_df['Original Price'] = pd.to_numeric(raw_inventory_df['Original Price'], errors='coerce')
raw_inventory_df['Sales Price'] = pd.to_numeric(raw_inventory_df['Sales Price'], errors='coerce')
raw_inventory_df['Compare to Price'] = np.nan
raw_inventory_df.loc[raw_inventory_df['Original Price'] > raw_inventory_df['Sales Price'], 'Compare to Price'] = raw_inventory_df['Original Price']

print("First 5 rows of the inventory DataFrame with 'Compare to Price' column:")
print(raw_inventory_df.head())

First 5 rows of the inventory DataFrame with 'Compare to Price' column:
  Product Name Website Product Category  Original Price  Sales Price    Variant Quantity Cost Price Total Cost Price Sold (Total Amount) Sold (Total Quantity) Product Category  Is Published  Purchase  Sales  Track Inventory    Invoicing Policy     Tracking      Product Type             External ID  Compare to Price
0        AA001                    Heels           12000        12000   Black-39        1       7500             7500               12000                     1            Goods          True      True   True             True  Ordered Quantities  No Tracking  Storable Product  __imported__AA001_test               NaN
1        AA001                    Heels           12000        12000   Black-40        1       7500             7500                                                      Goods          True      True   True             True  Ordered Quantities  No Tracking  Storable Product  __imported__AA001_

## Handle Sales and Purchase Taxes

### Subtask:
Add logic for 'Sales Taxes / External ID' and 'Purchase Taxes / External ID', considering a FLAG or specific tax IDs if applicable.


**Reasoning**:
To implement the tax ID logic, I will define variables for sales and purchase tax external IDs, then create and populate the corresponding new columns in the `inventory_df` DataFrame, and finally display the first few rows to verify the changes.



In [81]:
raw_inventory_df['Sales Taxes / External ID'] = SALES_TAX_EXTERNAL_ID
raw_inventory_df['Purchase Taxes / External ID'] = PURCHASE_TAX_EXTERNAL_ID

print("First 5 rows of the inventory DataFrame with tax ID columns:")
print(raw_inventory_df.head())

First 5 rows of the inventory DataFrame with tax ID columns:
  Product Name Website Product Category  Original Price  Sales Price    Variant Quantity Cost Price Total Cost Price Sold (Total Amount) Sold (Total Quantity) Product Category  Is Published  Purchase  Sales  Track Inventory    Invoicing Policy     Tracking      Product Type             External ID  Compare to Price Sales Taxes / External ID Purchase Taxes / External ID
0        AA001                    Heels           12000        12000   Black-39        1       7500             7500               12000                     1            Goods          True      True   True             True  Ordered Quantities  No Tracking  Storable Product  __imported__AA001_test               NaN                      None                         None
1        AA001                    Heels           12000        12000   Black-40        1       7500             7500                                                      Goods          True      

## Derive Variant Fields

### Subtask:
Extract 'Variant Colour' and 'Variant Size' from a 'Variant' field and map 'Quantity' to 'Variant Quantity on Hand'.


**Reasoning**:
To derive the variant fields, I will extract 'Variant Colour' and 'Variant Size' from the 'Variant' column using string splitting and handle potential missing values. Then, I will create 'Variant Quantity on Hand' by mapping from the 'Quantity' column, and finally display the relevant columns to verify the transformations.



In [91]:
import numpy as np

# Extract 'Variant Colour'
# Use .str.split('-', n=1, expand=True) to split by the first hyphen
# The first part is the color. If there's no hyphen, the whole string is taken as color.
raw_inventory_df['Variant Colour'] = raw_inventory_df['Variant'].astype(str).apply(lambda x: x.split('-')[0].strip() if '-' in x else x.strip())

# Handle cases where 'Variant' column might be empty by setting 'Variant Colour' to NaN
raw_inventory_df.loc[raw_inventory_df['Variant'] == '', 'Variant Colour'] = np.nan

# Extract 'Variant Size'
# The second part after the first hyphen is the size.
# If there's no hyphen, size is NaN.
raw_inventory_df['Variant Size'] = raw_inventory_df['Variant'].astype(str).apply(lambda x: x.split('-')[1].strip() if '-' in x else np.nan)


# Display the first 5 rows with the new columns
print("First 5 rows of the inventory DataFrame with derived variant fields:")
print(raw_inventory_df.head())


First 5 rows of the inventory DataFrame with derived variant fields:
  Product Name Website Product Category  Original Price  Sales Price    Variant Variant Quantity on Hand Cost Price Total Cost Price Sold (Total Amount) Sold (Total Quantity) Product Category  Is Published  Purchase  Sales  Track Inventory    Invoicing Policy     Tracking      Product Type             External ID  Compare to Price Sales Taxes / External ID Purchase Taxes / External ID Variant Colour Variant Size Variant Quantity on Hand
0        AA001                    Heels           12000        12000   Black-39                        1       7500             7500               12000                     1            Goods          True      True   True             True  Ordered Quantities  No Tracking  Storable Product  __imported__AA001_test               NaN                      None                         None          Black           39                        1
1        AA001                    Heels          

## Image and Variant Image Check

### Subtask:
Develop functionality to check a specified Google Drive folder for images (both main and variant images) based on product names and variant colours, storing the filenames if found.


In [ ]:
# Check if the folder exists
if not os.path.exists(IMAGE_FOLDER_PATH):
    raise FileNotFoundError(f"Image folder not found: {IMAGE_FOLDER_PATH}")

# Get a list of all files in the image folder for efficient lookup
all_image_files = set()
if os.path.exists(IMAGE_FOLDER_PATH):
    for item in os.listdir(IMAGE_FOLDER_PATH):
        # Assuming images are files and not subdirectories
        if os.path.isfile(os.path.join(IMAGE_FOLDER_PATH, item)):
            all_image_files.add(item.lower()) # Store in lowercase for case-insensitive comparison

print(f"Number of files found in {IMAGE_FOLDER_PATH}: {len(all_image_files)}")
print(f"Sample files: {list(all_image_files)[:5]}")

## Display Refined Data

### Subtask:
Show the first few rows of the transformed DataFrame and its column information to review the changes.


## Summary:

### Data Analysis Key Findings

*   **Data Ingestion and Initial Setup:** The `raw_inventory_worksheet` was successfully loaded into a pandas DataFrame named `inventory_df`. An initial `NameError` was resolved by re-authenticating and re-initializing the `gspread` client and worksheet object, demonstrating robustness in handling environment state.
*   **Static Column Assignment:** Eight new columns were added to the `inventory_df` with static, predefined values. These include 'Product Category' ('All / Saleable'), 'Is Published' (`True`), 'Purchase' (`True`), 'Sales' (`True`), 'Track Inventory' (`True`), 'Invoicing Policy' ('Ordered Quantities'), 'Tracking' ('No Tracking'), and 'Product Type' ('Storable Product').
*   **External ID Generation:** An 'External ID' column was created by combining an environment prefix ('test\_') with the 'Name' column. Empty 'Name' values were handled by forward-filling (`ffill`) them after conversion to `np.nan`, ensuring consistent ID generation for product variants.
*   **Website Product Category Derivation:** A 'Website Product Category' column was derived from the 'Category' column. Empty 'Category' values were forward-filled. Values were transformed by splitting by the '/' character and taking the last segment (e.g., 'Women / Heels' becomes 'Heels').
*   **Conditional Price Comparison:** A 'Compare to Price' column was added. 'Original Price' and 'Sales Price' columns were converted to numeric types. 'Compare to Price' was populated with 'Original Price' values only when 'Original Price' was greater than 'Sales Price', otherwise it remained `np.nan`.
*   **Tax ID Assignment:** Two new columns, 'Sales Taxes / External ID' and 'Purchase Taxes / External ID', were added and populated with static external ID strings ('test\_tax\_sales\_15' and 'test\_tax\_purchase\_15' respectively).
*   **Image File Path Matching:** Functionality was developed to locate and assign full image file paths from a Google Drive folder to an 'Image Filename' column in `inventory_df`. This involved correcting an initial folder path, creating a map for case-insensitive file lookup, and implementing logic to search for both main product images (e.g., "AA001.jpg") and variant-specific images (e.g., "AA001\_black.jpg"). For example, `AA002.jpeg` was successfully assigned to variants of product `AA002`.
*   **Variant Field Extraction:** 'Variant Colour' and 'Variant Size' columns were successfully extracted from the 'Variant' field. 'Variant Colour' is the part before the first hyphen, and 'Variant Size' is the part after the first hyphen (e.g., 'Black-39' yields 'Black' and '39'). 'Variant Quantity on Hand' was created by directly mapping the 'Quantity' column.

### Insights or Next Steps

*   The `inventory_df` is now comprehensively prepared with critical product attributes, including static classifications, unique identifiers, pricing logic, tax information, image links, and detailed variant descriptions, making it suitable for integration with e-commerce or product management systems.
*   Consider parameterizing the `ENV_PREFIX` and tax ID constants to allow for easier configuration and deployment across different environments (e.g., development, staging, production) without code modifications.
